# LIFE formation experiments — STK visualisation

Re-create the **single transfer**, **Rotating baseline**, **Fixed baseline**, and **re-pointing** from `LIFE_formation_first_steps.ipynb`, and watch their **Astrogator-propagated spacecraft in STK**. This notebook keeps the embedded Base state, force-model component, reference geometry, impulsive transfer solver and endpoint checks. It omits the Python plots, HTML movies, tracking-error reports and Δv comparison study.

**Run sections 1–12 in order. Section 12 starts the first available experiment automatically. Then run the individual viewing cells in sections 13–20 as needed.** Run All also works: the final cell returns to the first available experiment and starts it again. Replaying, changing views or toggling vectors does not re-run the targeter.

Requires **Windows, STK 13 Desktop with Astrogator, the `agi.stk13` Python API, NumPy and SciPy** in the notebook's kernel. It starts a **new STK application** and creates its own scenario. No previous scenario, helper file or JWST ephemeris is required. The separate optional long-term/JWST diagnostics in the analysis notebook are outside this formation viewer.

All experiments start independently at `T0`, by default **6 October 2022, 07:10 UTC**; they are alternatives, not successive mission phases. Only the chosen experiment and burn spacing are visible together. Base is additional to the selected followers. The trajectories and separations are physical; the screen markers are enlarged symbols.

The Base-centred display frame translates with Base and retains **Earth ICRF axes**. It is not a rotating L2-frame formation. A fixed baseline therefore looks nearly stationary in a close-up; a rotation is commanded reference motion, rather than a passive consequence of orbit dynamics. Impulses change velocity instantaneously and do not produce a visible engine plume.

## 0A. Reconnect to a saved experiment

Open the saved scenario in STK first. Set `STATE_FILE` to its companion `viewer_state.json`, then run this cell to restore the notebook's viewing variables.

Afterward, run Step 11 and 12. The viewing cells and 10A–10D can then be used normally. Skip Steps 1–10 when reopening an experiment.

In [1]:
"""from pathlib import Path
from itertools import combinations
import json
import numpy as np
from agi.stk13.stkdesktop import STKDesktop

STATE_FILE = Path(r'C:\your\saved\scenario\folder\viewer_state.json')

state = json.loads(STATE_FILE.read_text(encoding='utf-8'))
stk = STKDesktop.AttachToApplication()
root = stk.Root
scenario = root.CurrentScenario

if scenario is None or scenario.InstanceName != state['scenario_name']:
    raise RuntimeError('Open the matching saved scenario in STK first.')

for dimension, value in [('Distance', 'm'), ('SmallDistance', 'm'),
                         ('Time', 'sec'), ('DateFormat', 'EpSec')]:
    root.UnitPreferences.SetCurrentUnit(dimension, value)

root.ExecuteCommand('Units_SetConnect / Distance Meter Time Second Date GregorianUTC')

globals().update(state['settings'])
OUT = STATE_FILE.parent
BASE_PATH = state['base_path']
BASE_OBJECT = '*/' + BASE_PATH
LOCAL_SYSTEM = state['local_system']
Base = root.GetObjectFromPath(BASE_PATH)
T0 = float(state['T0'])
Q0 = np.asarray(state['Q0'], dtype=float)
LOS_START = np.asarray(state['LOS_START'], dtype=float)
SHOW_BASE = state['SHOW_BASE']
CURRENT_VIEW = state['CURRENT_VIEW']
ACTIVE_KEY = None
DAY_S = 86400.0


def cmd(command):
    try:
        return root.ExecuteCommand(command)
    except Exception as exc:
        raise RuntimeError('STK Connect failed: ' + command) from exc


def path_of(sat):
    return 'Satellite/' + sat.InstanceName


def unit(vector):
    vector = np.asarray(vector, dtype=float)
    return vector / np.linalg.norm(vector)


def base_state(epoch_s):
    system = root.CentralBodies.Item('Earth').Vgt.Systems.Item('ICRF')
    result = Base.Vgt.Points.Item('Center').LocateInSystemWithRate(float(epoch_s), system)
    if not result.IsValid:
        raise RuntimeError('No saved Base state at the requested time.')
    r, v = result.Position, result.Velocity
    return np.array([r.X, r.Y, r.Z]), np.array([v.X, v.Y, v.Z])


clips = {}
for saved_clip in state['clips']:
    key = tuple(saved_clip['key'])
    group = []
    for saved_case in saved_clip['cases']:
        case = dict(saved_case)
        case['satellite'] = root.GetObjectFromPath(case.pop('satellite_path'))
        case['waypoints'] = np.asarray(case['waypoints'], dtype=float)
        group.append(case)
    clips[key] = group

all_cases = [case for group in clips.values() for case in group]
vector_displays = state['vector_displays']
edge_displays = state['edge_displays']

for item in vector_displays:
    if isinstance(item['category'], list):
        item['category'] = tuple(item['category'])

for item in edge_displays:
    item['key'] = tuple(item['key'])


def apply_overlays():
    for item in vector_displays:
        category = item['category']
        show = SHOW_VECTORS and (
            (category == 'Initial_LOS' and SHOW_LOS_GUIDES)
            or (category == 'Final_LOS' and SHOW_LOS_GUIDES and ACTIVE_KEY is not None and ACTIVE_KEY[0] == 'repointing')
            or (category == 'axes' and SHOW_ICRF_AXES)
            or (isinstance(category, tuple) and category == ACTIVE_KEY and SHOW_OFFSET_VECTORS))
        if CURRENT_VIEW == 'context':
            show = False
        cmd(f'VO {item["owner"]} SetVectorGeometry Modify "{item["component"]}" Show {"On" if show else "Off"}')

    for item in edge_displays:
        show = SHOW_EDGES and item['key'] == ACTIVE_KEY
        cmd(f'VO * ObjectLine Modify FromObj {item["a"]} ToObj {item["b"]} Show {"On" if show else "Off"} WindowID {WINDOW_ID}')


print('Reconnected:', scenario.InstanceName)
print('Available clips:', list(clips))"""

'from pathlib import Path\nfrom itertools import combinations\nimport json\nimport numpy as np\nfrom agi.stk13.stkdesktop import STKDesktop\n\nSTATE_FILE = Path(r\'C:\\your\\saved\\scenario\x0colder\x0biewer_state.json\')\n\nstate = json.loads(STATE_FILE.read_text(encoding=\'utf-8\'))\nstk = STKDesktop.AttachToApplication()\nroot = stk.Root\nscenario = root.CurrentScenario\n\nif scenario is None or scenario.InstanceName != state[\'scenario_name\']:\n    raise RuntimeError(\'Open the matching saved scenario in STK first.\')\n\nfor dimension, value in [(\'Distance\', \'m\'), (\'SmallDistance\', \'m\'),\n                         (\'Time\', \'sec\'), (\'DateFormat\', \'EpSec\')]:\n    root.UnitPreferences.SetCurrentUnit(dimension, value)\n\nroot.ExecuteCommand(\'Units_SetConnect / Distance Meter Time Second Date GregorianUTC\')\n\nglobals().update(state[\'settings\'])\nOUT = STATE_FILE.parent\nBASE_PATH = state[\'base_path\']\nBASE_OBJECT = \'*/\' + BASE_PATH\nLOCAL_SYSTEM = state[\'local_

## 1. Experiment and display settings

Set `NUM_SPACECRAFT = 6` for the complete octahedron. Set `TRACK_DURATION_DAYS = None` for a full rotation; this means many more targeter solves. The build in section 8 is the slow step.

`SHOW_VECTORS` is the master arrow switch. The individual switches choose which arrows appear. `SHOW_EDGES` controls connecting lines separately. Change these display switches and re-run section 18 without rebuilding the dynamics. Change physical inputs only before restarting and running from the top.

`PLAYBACK_SECONDS` is the approximate wall-clock length of a full clip, assuming STK can render at `PLAYBACK_FPS`. The notebook uses a fixed simulation-time step and loops only within the selected experiment's time interval. `TRAIL_MODE = 'history'` draws the path as time passes; `'full'` shows the complete path from the start.

In [2]:
from pathlib import Path
from datetime import datetime
from functools import lru_cache
from itertools import combinations
import json
import os
import numpy as np
from scipy.spatial.transform import Rotation

NUM_SPACECRAFT = 6
SPACECRAFT_LABELS = None                 # E.g. ['Xp', 'Zp'] with NUM_SPACECRAFT = 2.
BURN_SPACINGS_HOURS = [24.0]
ROTATION_PERIOD_DAYS = 28.0
TRACK_DURATION_DAYS = None                 # None = one complete rotation.
L_M = 1000.0                             # Base-to-vertex distance [m].
LOS_START = np.array([0.0, 0.0, 1.0])
LOS_END = np.array([np.sin(np.deg2rad(20)), 0.0, np.cos(np.deg2rad(20))])
SLEW_DURATION_DAYS = 2.0
START_OFFSET_DAYS = 0.0
RUN_SINGLE_TRANSFER = False
RUN_ROTATING_BASELINE = True
RUN_FIXED_BASELINE = False
RUN_REPOINTING = False
STOP_AT_END = True

PROPAGATOR_NAME = 'Earth Default High Fidelity v13'
BASE_SEED_UTCG = '6 Oct 2022 07:10:00.000'
BASE_SEED_R_M = np.array([1.15803168e9, 9.39831792e7, 4.32308211e8])
BASE_SEED_V_MPS = np.array([-124.13911218, 510.44906145, 237.63841329])
DRY_MASS_KG = 50.0
FUEL_MASS_KG = 50.0
AREA_M2 = 2.0
CR = 1.0
CD = 2.2
POSITION_TOL_M = 0.10
VELOCITY_TOL_MPS = 1e-6
DC_PERTURBATION_MPS = 1e-4
DC_MAX_STEP_MPS = 0.1
DC_MAX_ITERATIONS = 30

WINDOW_ID = 1                            # Number in STK's 3D Graphics window title.
CAMERA_RANGE_M = 8.0 * L_M
PLAYBACK_SECONDS = 40.0
PLAYBACK_FPS = 20.0
LOOP_PLAYBACK = True
AUTO_PLAY = True
TRAIL_MODE = 'history'                    # 'history' or 'full'.
SHOW_VECTORS = True
SHOW_OFFSET_VECTORS = True               # Actual Base-to-follower arrows, true length.
SHOW_LOS_GUIDES = True                   # Initial LOS; final LOS also shown for re-pointing.
SHOW_ICRF_AXES = False                   # ICRF directions drawn at Base, display-scaled.
SHOW_EDGES = True                        # Only actual edges between selected vertices.
SHOW_LABELS = True
SAVE_SCENARIO = True

COLOURS = {'Xp':'red', 'Xm':'cyan', 'Yp':'lime', 'Ym':'magenta',
           'Zp':'yellow', 'Zm':'orange'}


## 2. Check the settings and define the geometry

The same six vertex labels and reference equations are used as in the analysis notebook. Four equatorial vertices rotate about `LOS_START`; the two axial vertices stay on that axis during the Rotating baseline. During re-pointing, the entire reference octahedron slews smoothly from `LOS_START` to `LOS_END`. Only selected vertices become simulated spacecraft.

The reference velocities define the arrival velocity targets. As in the source notebook, each follower is released with Base's translational velocity; the targeter finds its first departure impulse. With `STOP_AT_END`, the final arrival matches Base's velocity.

In [3]:
LABELS = ['Xp', 'Xm', 'Yp', 'Ym', 'Zp', 'Zm']
if type(NUM_SPACECRAFT) is not int or not 1 <= NUM_SPACECRAFT <= 6:
    raise ValueError('NUM_SPACECRAFT must be an integer from 1 to 6.')
SELECTED_LABELS = LABELS[:NUM_SPACECRAFT] if SPACECRAFT_LABELS is None else list(SPACECRAFT_LABELS)
if (len(SELECTED_LABELS) != NUM_SPACECRAFT or len(set(SELECTED_LABELS)) != NUM_SPACECRAFT
        or any(x not in LABELS for x in SELECTED_LABELS)):
    raise ValueError('Select NUM_SPACECRAFT different labels from ' + str(LABELS))
SELECTED_VERTICES = [LABELS.index(x) for x in SELECTED_LABELS]
if not BURN_SPACINGS_HOURS or any(not np.isfinite(x) or x < 1/3600 for x in BURN_SPACINGS_HOURS):
    raise ValueError('Choose positive burn spacings of at least one second.')
BURN_SPACINGS_HOURS = sorted(set(float(x) for x in BURN_SPACINGS_HOURS))
for name in ['L_M', 'ROTATION_PERIOD_DAYS', 'SLEW_DURATION_DAYS', 'CAMERA_RANGE_M',
             'PLAYBACK_SECONDS', 'PLAYBACK_FPS', 'POSITION_TOL_M', 'VELOCITY_TOL_MPS',
             'DC_PERTURBATION_MPS', 'DC_MAX_STEP_MPS', 'DRY_MASS_KG']:
    if not np.isfinite(globals()[name]) or globals()[name] <= 0:
        raise ValueError(name + ' must be positive and finite.')
if TRACK_DURATION_DAYS is not None and (not np.isfinite(TRACK_DURATION_DAYS) or TRACK_DURATION_DAYS <= 0):
    raise ValueError('TRACK_DURATION_DAYS must be positive or None.')
if not np.isfinite(START_OFFSET_DAYS) or START_OFFSET_DAYS < 0:
    raise ValueError('START_OFFSET_DAYS must be non-negative.')
if any(not np.isfinite(x) or x < 0 for x in [FUEL_MASS_KG, AREA_M2, CR, CD]):
    raise ValueError('Mass, area and optical coefficients must be non-negative.')
if DC_MAX_ITERATIONS < 1 or WINDOW_ID < 1:
    raise ValueError('Iteration count and window ID must be positive.')
if TRAIL_MODE not in ['history', 'full']:
    raise ValueError("TRAIL_MODE must be 'history' or 'full'.")
for name in ['BASE_SEED_R_M', 'BASE_SEED_V_MPS', 'LOS_START', 'LOS_END']:
    if np.shape(globals()[name]) != (3,) or not np.isfinite(globals()[name]).all():
        raise ValueError(name + ' must have three finite components.')
if not any([RUN_SINGLE_TRANSFER, RUN_ROTATING_BASELINE, RUN_FIXED_BASELINE, RUN_REPOINTING]):
    raise ValueError('Enable at least one experiment.')
DAY_S = 86400.0
ROTATION_PERIOD_S = ROTATION_PERIOD_DAYS * DAY_S
TRACK_DURATION_S = (ROTATION_PERIOD_DAYS if TRACK_DURATION_DAYS is None else TRACK_DURATION_DAYS) * DAY_S
SLEW_DURATION_S = SLEW_DURATION_DAYS * DAY_S
DEMO_SPACING_S = max(BURN_SPACINGS_HOURS) * 3600.0
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = Path.cwd() / 'results' / ('LIFE_STK_visualisation_' + RUN_TAG)
OUT.mkdir(parents=True, exist_ok=False)
VERTICES = L_M * np.array([[1,0,0], [-1,0,0], [0,1,0], [0,-1,0], [0,0,1], [0,0,-1]], float)
PAIRS = list(combinations(range(6), 2))

def unit(vector):
    vector = np.asarray(vector, dtype=float)
    length = np.linalg.norm(vector)
    if length < 1e-14:
        raise ValueError('A direction must be nonzero.')
    return vector / length

def initial_orientation(los):
    z = unit(los)
    seed = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(seed, z)) > 0.95:
        seed = np.array([0.0, 1.0, 0.0])
    x = unit(seed - np.dot(seed, z) * z)
    y = np.cross(z, x)
    return np.column_stack([x, y, z])

Q0 = initial_orientation(LOS_START)
_slew_cross = np.cross(unit(LOS_START), unit(LOS_END))
SLEW_ANGLE = np.arctan2(np.linalg.norm(_slew_cross), np.dot(unit(LOS_START), unit(LOS_END)))
if np.isclose(SLEW_ANGLE, np.pi):
    raise ValueError('Choose an explicit slew axis for a 180-degree re-pointing.')
SLEW_AXIS = unit(_slew_cross) if np.linalg.norm(_slew_cross) > 1e-14 else Q0[:, 0]
print('Centre-to-vertex:', L_M, 'm; edge:', np.sqrt(2)*L_M, 'm; opposite:', 2*L_M, 'm')

def reference(t_s, operation='Rotating baseline'):
    t = float(t_s)
    if operation == 'Rotating baseline':
        omega = 2*np.pi / ROTATION_PERIOD_S * unit(LOS_START)
        turn = Rotation.from_rotvec(omega*t).as_matrix()
    elif operation == 'repointing':
        u = np.clip(t / SLEW_DURATION_S, 0.0, 1.0)
        s = 10*u**3 - 15*u**4 + 6*u**5
        ds = (30*u**2 - 60*u**3 + 30*u**4) / SLEW_DURATION_S
        turn = Rotation.from_rotvec(SLEW_AXIS*SLEW_ANGLE*s).as_matrix()
        omega = SLEW_AXIS*SLEW_ANGLE*ds
    elif operation == 'Fixed baseline':
        turn = np.eye(3)
        omega = np.zeros(3)
    else:
        raise ValueError(operation)
    position_m = VERTICES @ (turn @ Q0).T
    velocity_mps = np.cross(omega, position_m)
    return position_m, velocity_mps

print('Follower colours:', {x: COLOURS[x] for x in SELECTED_LABELS})
print('Base: white. Base is additional to the followers.')
print('Rotation arc:', 360 * TRACK_DURATION_S / ROTATION_PERIOD_S, 'degrees')
print('Scenario output folder:', OUT)


Centre-to-vertex: 1000.0 m; edge: 1414.213562373095 m; opposite: 2000.0 m
Follower colours: {'Xp': 'red', 'Xm': 'cyan', 'Yp': 'lime', 'Ym': 'magenta', 'Zp': 'yellow', 'Zm': 'orange'}
Base: white. Base is additional to the followers.
Rotation arc: 360.0 degrees
Scenario output folder: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Own-Code\First-Experiment\results\LIFE_STK_visualisation_20261102_233558_234649


## 3. Open a fresh STK Desktop scenario

This launches a separate STK application, rather than attaching to an existing scenario. Object Model units are metres, seconds, kg and epoch seconds. Connect units are set separately to metres, seconds and UTC calendar dates, so camera distances and playback times have explicit units.

Each experiment uses `T0`; Base is propagated from the embedded seed epoch. No orbit-maintenance controller is added to Base, and this short arc does not establish long-term halo stability.

In [4]:
if os.name != 'nt':
    raise RuntimeError('The following cells require Windows STK 13 Desktop and Astrogator.')
from agi.stk13.stkdesktop import STKDesktop
from agi.stk13.stkobjects import AgESTKObjectType, AgEVePropagatorType
from agi.stk13.stkobjects.astrogator import (
    AgEVASegmentType, AgEVAElementType, AgEVAManeuverType,
    AgEVAAttitudeControl, AgEVAControlManeuver, AgEVATargetSeqAction,
    AgEVAProfileMode,
)
stk = STKDesktop.StartApplication(visible=True, userControl=True)
root = stk.Root
root.NewScenario('LIFE_Visual_' + RUN_TAG)
scenario = root.CurrentScenario
root.UnitPreferences.ResetUnits()
root.UnitPreferences.SetCurrentUnit('DateFormat', 'UTCG')
scenario.Epoch = BASE_SEED_UTCG
for dimension, value in [('Distance','m'), ('SmallDistance','m'), ('Time','sec'),
                         ('DateFormat','EpSec'), ('Mass','kg')]:
    root.UnitPreferences.SetCurrentUnit(dimension, value)
SEED_EPOCH = float(root.ConversionUtility.ConvertDate('UTCG', 'EpSec', BASE_SEED_UTCG))
T0 = SEED_EPOCH + START_OFFSET_DAYS*DAY_S
durations = []
if RUN_SINGLE_TRANSFER: durations.append(DEMO_SPACING_S)
if RUN_ROTATING_BASELINE or RUN_FIXED_BASELINE: durations.append(TRACK_DURATION_S)
if RUN_REPOINTING: durations.append(SLEW_DURATION_S)
REQUIRED_DURATION_S = max(durations)
BASE_END_EPOCH = T0 + REQUIRED_DURATION_S + 3600.0
scenario.SetTimePeriod(str(SEED_EPOCH), str(BASE_END_EPOCH))
root.ExecuteCommand('Units_SetConnect / Distance Meter Time Second Date GregorianUTC')
FRAME = 'CentralBody/Earth ICRF'
print('Fresh scenario:', scenario.InstanceName)
print('Follower release UTC:', root.ConversionUtility.ConvertDate('EpSec','UTCG',str(T0)))


Fresh scenario: LIFE_Visual_20261102_233558_234649
Follower release UTC: 6 Oct 2022 07:10:00.000


## 4. Define Astrogator spacecraft, coast segments and the Base trajectory

These constructors retain the source notebook's initial-state fields, spacecraft properties and named propagator. Segment-specific trajectory colours are disabled so each spacecraft keeps its own colour through every coast and impulse. Temporary targeter trajectories are disabled; the final propagated ephemeris supplies the displayed paths.

The new `Visual_Base_ICRF` system has its origin at Base and Earth ICRF axis directions. It is used only for drawing trajectories; the numerical targeter still operates in Earth-centred ICRF.

In [5]:
def new_experiment(label, position_m, velocity_mps, start_epoch=None, frame=FRAME):
    if start_epoch is None:
        start_epoch = T0
    name = 'FS_' + label
    sat = scenario.Children.New(AgESTKObjectType.eSatellite, name)
    sat.SetPropagatorType(AgEVePropagatorType.ePropagatorAstrogator)
    driver = sat.Propagator
    driver.Options.EnableTrajectorySegmentColors = False
    driver.Options.DrawTrajectoryIn3D = False
    driver.Options.DrawTrajectoryIn2D = False
    driver.MainSequence.RemoveAll()
    initial = driver.MainSequence.Insert(AgEVASegmentType.eVASegmentTypeInitialState, 'Initial', '-')
    initial.CoordSystemName = frame
    initial.OrbitEpoch = str(float(start_epoch))
    initial.SetElementType(AgEVAElementType.eVAElementTypeCartesian)
    for attr, value in zip(['X','Y','Z','Vx','Vy','Vz'], np.r_[position_m,velocity_mps]):
        setattr(initial.Element, attr, float(value))
    properties = dict(Cd=CD, Cr=CR, DryMass=DRY_MASS_KG, DragArea=AREA_M2,
                      K1=1.0, K2=1.0, RadiationPressureArea=AREA_M2,
                      Ck=1.0, SolarRadiationPressureArea=AREA_M2)
    for attr, value in properties.items():
        setattr(initial.SpacecraftParameters, attr, float(value))
    for attr, value in dict(FuelMass=FUEL_MASS_KG, FuelDensity=1000.0,
                            TankPressure=5000.0, TankTemperature=293.15).items():
        setattr(initial.FuelTank, attr, value)
    return sat, driver

def add_coast(sequence, name, duration_s):
    segment = sequence.Insert(AgEVASegmentType.eVASegmentTypePropagate, name, '-')
    segment.PropagatorName = PROPAGATOR_NAME
    stops = segment.StoppingConditions
    if 'Duration' not in [stops.Item(i).Name for i in range(stops.Count)]:
        stops.Add('Duration')
    for i in range(stops.Count-1, -1, -1):
        if stops.Item(i).Name != 'Duration':
            stops.Remove(i)
    duration = stops.Item('Duration')
    duration.Active = True
    duration.Properties.Trip = float(duration_s)
    duration.Properties.Tolerance = 1e-6
    duration.Properties.Sequence = 'Stop'
    segment.MinPropagationTime = 0.0
    segment.EnableMaxPropagationTime = True
    segment.MaxPropagationTime = float(duration_s) + 60.0
    return segment


Base, base_driver = new_experiment('Base', BASE_SEED_R_M, BASE_SEED_V_MPS, SEED_EPOCH)
base_coast = add_coast(base_driver.MainSequence, 'Reference_coast', BASE_END_EPOCH-SEED_EPOCH)
base_driver.RunMCS()
BASE_PATH = 'Satellite/' + Base.InstanceName
BASE_OBJECT = '*/' + BASE_PATH
root.ExecuteCommand(f'VectorTool * {BASE_PATH} Create System Visual_Base_ICRF "Assembled" "{BASE_PATH} Center" "CentralBody/Earth ICRF"')
LOCAL_SYSTEM = BASE_PATH + ' Visual_Base_ICRF'
CALC_NAMES = {FRAME: []}
for axis in 'XYZ':
    name = axis + '_FS_ICRF'
    root.ExecuteCommand(f'ComponentBrowser */ Duplicate "Calculation Objects" {axis} "{name}"')
    root.ExecuteCommand(f'ComponentBrowser */ SetValue "Calculation Objects" "{name}" CoordSystem "{FRAME}"')
    CALC_NAMES[FRAME].append(name)
print('Base propagated. Base-centred inertial display frame created.')


Base propagated. Base-centred inertial display frame created.


## 5. Read states and define waypoint times

Read Base's position and velocity in Earth ICRF, and check the embedded initial state. A waypoint is a desired follower position and velocity at a specific time. Each coast ends at a waypoint; its duration is the chosen burn spacing, with a shorter last leg if needed.

In [6]:
earth = root.CentralBodies.Item('Earth')
icrf_system = earth.Vgt.Systems.Item('ICRF')

def xyz(vector):
    return np.array([vector.X, vector.Y, vector.Z], dtype=float)

def point_state(satellite, epoch_s, system=icrf_system):
    result = satellite.Vgt.Points.Item('Center').LocateInSystemWithRate(float(epoch_s), system)
    if not result.IsValid:
        raise RuntimeError(f'No valid state for {satellite.InstanceName} at EpSec {epoch_s}.')
    position, velocity = xyz(result.Position), xyz(result.Velocity)
    if not np.isfinite(np.r_[position,velocity]).all():
        raise RuntimeError(f'Non-finite VGT state for {satellite.InstanceName} at EpSec {epoch_s}.')
    return position, velocity

@lru_cache(maxsize=200000)
def base_state(epoch_s):
    return point_state(Base, float(epoch_s))

def state_vectors(state, frame=FRAME):
    converted = state.GetInFrameName(frame)
    converted.SetElementType(AgEVAElementType.eVAElementTypeCartesian)
    e = converted.Element
    position = np.array([e.X,e.Y,e.Z], dtype=float)
    velocity = np.array([e.Vx,e.Vy,e.Vz], dtype=float)
    if not np.isfinite(np.r_[position,velocity]).all():
        raise RuntimeError('STK returned a non-finite state in ' + frame)
    return position, velocity

def segment_state(segment, frame=FRAME):
    return state_vectors(segment.FinalState, frame)

for epoch in [SEED_EPOCH, T0, T0+REQUIRED_DURATION_S, BASE_END_EPOCH]:
    base_state(float(epoch))
np.testing.assert_allclose(base_state(SEED_EPOCH)[0], BASE_SEED_R_M, rtol=0, atol=1e-3)
np.testing.assert_allclose(base_state(SEED_EPOCH)[1], BASE_SEED_V_MPS, rtol=0, atol=VELOCITY_TOL_MPS)
print('Base start [m], [m/s]:', base_state(T0))

def waypoint_times(duration_s, spacing_s):
    if duration_s <= 0 or spacing_s <= 0:
        raise ValueError('Duration and spacing must be positive.')
    times = np.arange(0.0, duration_s, spacing_s)
    times = times[times < duration_s - 1e-7]
    return np.r_[times, duration_s]


Base start [m], [m/s]: (array([1.15803168e+09, 9.39831792e+07, 4.32308211e+08]), array([-124.13911218,  510.44906145,  237.63841329]))


## 6. Retain the differential corrector and endpoint checks

The source notebook's impulse, target-sequence and leg-solver functions are retained. A departure impulse is solved to reach the next position; a separate arrival impulse matches the desired velocity. Position, velocity and arrival-time checks remain active. An unconverged leg stops the build rather than being shown as a valid result.

In [7]:
def add_impulse(sequence, name, dv_mps=(0,0,0)):
    segment = sequence.Insert(AgEVASegmentType.eVASegmentTypeManeuver, name, '-')
    segment.SetManeuverType(AgEVAManeuverType.eVAManeuverTypeImpulsive)
    segment.Maneuver.SetAttitudeControlType(AgEVAAttitudeControl.eVAAttitudeControlThrustVector)
    segment.Maneuver.UpdateMass = False
    segment.Maneuver.AttitudeControl.ThrustAxesName = FRAME
    segment.Maneuver.AttitudeControl.AssignCartesian(*map(float, dv_mps))
    return segment

def make_target_leg(driver, leg_number, duration_s, desired_position_m, frame, tolerance_m):
    ts = driver.MainSequence.Insert(AgEVASegmentType.eVASegmentTypeTargetSequence, f'Target_{leg_number:03d}', '-')
    ts.ContinueOnFailure = False
    departure = add_impulse(ts.Segments, 'Departure')
    coast = add_coast(ts.Segments, 'Coast', duration_s)
    dc = ts.Profiles.Item('Differential Corrector')
    dc.Mode = AgEVAProfileMode.eVAProfileModeIterate
    dc.MaxIterations = DC_MAX_ITERATIONS
    dc.EnableDisplayStatus = False
    controls = [AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianX,
                AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianY,
                AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianZ]
    for axis, control_enum in zip('XYZ', controls):
        departure.EnableControlParameter(control_enum)
        control = dc.ControlParameters.GetControlByPaths('Departure', f'ImpulsiveMnvr.Cartesian.{axis}')
        control.Enable = True
        control.Perturbation = DC_PERTURBATION_MPS
        control.MaxStep = DC_MAX_STEP_MPS
    for name, desired in zip(CALC_NAMES[frame], desired_position_m):
        coast.Results.Add('Cartesian Elems/' + name)
        result = dc.Results.GetResultByPaths('Coast', name)
        result.Enable = True
        result.DesiredValue = float(desired)
        result.Tolerance = tolerance_m
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunNominalSeq
    return ts, departure, coast, dc

def solve_leg(driver, number, dt, desired_r, desired_v=None, frame=FRAME, tolerance_m=POSITION_TOL_M):
    ts, departure, coast, dc = make_target_leg(driver, number, dt, desired_r, frame, tolerance_m)
    driver.RunMCS()
    nominal_r, _ = segment_state(coast, frame)
    calc_r = np.array([float(coast.GetResultValue(name)) for name in CALC_NAMES[frame]])
    np.testing.assert_allclose(calc_r, nominal_r, rtol=0, atol=0.005,
                               err_msg='Target calculation frame or units do not match the requested state frame.')
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunActiveProfiles
    driver.RunMCS()
    diagnostic = {'leg':number, 'profile_status':str(dc.Status), 'iterations':int(dc.NumIterations)}
    for axis, name in zip('XYZ', CALC_NAMES[frame]):
        result = dc.Results.GetResultByPaths('Coast', name)
        residual = float(result.CurrentValue-result.DesiredValue)
        diagnostic[f'target_residual_{axis}_m'] = residual
        if not np.isfinite(residual) or abs(residual) > tolerance_m*1.01:
            raise RuntimeError(f'Position target failed: {diagnostic}')
    ts.ApplyProfiles()
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunNominalSeq
    driver.RunMCS()
    r_end, _ = segment_state(coast, frame)
    if np.max(np.abs(r_end-desired_r)) > tolerance_m*1.05:
        raise RuntimeError('Applied nominal solution misses the position target.')
    dv_departure = np.array(departure.Maneuver.AttitudeControl.QueryCartesian(), dtype=float).ravel()
    velocity_before = state_vectors(departure.InitialState)[1]
    np.testing.assert_allclose(segment_state(departure)[1]-velocity_before, dv_departure,
                               rtol=0, atol=VELOCITY_TOL_MPS)
    expected_epoch = float(coast.InitialState.Epoch)+dt
    if abs(float(coast.FinalState.Epoch)-expected_epoch) > 1e-3:
        raise RuntimeError('Coast stopped at an unexpected epoch.')
    diagnostic['endpoint_position_error_m'] = float(np.linalg.norm(r_end-desired_r))
    arrival, dv_arrival = None, np.zeros(3)
    diagnostic['pre_arrival_velocity_error_mps'] = np.nan
    diagnostic['endpoint_velocity_error_mps'] = np.nan
    if desired_v is not None:
        # Velocity matching is always performed in Earth ICRF, regardless of position frame.
        v_end = segment_state(coast)[1]
        dv_arrival = desired_v-v_end
        arrival = add_impulse(driver.MainSequence, f'Arrival_{number:03d}', dv_arrival)
        driver.RunMCS()
        achieved_r = segment_state(arrival, frame)[0]
        achieved_v = segment_state(arrival)[1]
        if (np.max(np.abs(achieved_r-desired_r)) > tolerance_m*1.05
                or np.linalg.norm(achieved_v-desired_v) > VELOCITY_TOL_MPS):
            raise RuntimeError('Post-arrival state verification failed.')
        diagnostic['pre_arrival_velocity_error_mps'] = float(np.linalg.norm(v_end-desired_v))
        diagnostic['endpoint_velocity_error_mps'] = float(np.linalg.norm(achieved_v-desired_v))
    return departure, coast, arrival, dv_departure, dv_arrival, diagnostic


## 7. Define a lightweight visual experiment runner

This builds the same sequence of solved transfers, without sampling trajectories for Python plots or calculating a Δv comparison. It retains each burn vector and waypoint diagnostic for replay. The single-transfer demonstration uses the first selected vertex, the largest configured spacing, and no final stop relative to Base, as in the source notebook.

In [8]:
clips = {}

def number_tag(value):
    return format(float(value), '.12g').replace('.', 'p')

def run_visual_case(label, operation, vertex, duration_s, spacing_s, stop_at_end=STOP_AT_END):
    times = waypoint_times(duration_s, spacing_s)
    br0, bv0 = base_state(float(T0))
    p0, _ = reference(0, operation)
    sat, driver = new_experiment(label, br0+p0[vertex], bv0)
    burns, diagnostics = [], []
    for leg, (ta, tb) in enumerate(zip(times[:-1], times[1:]), 1):
        br, bv = base_state(float(T0+tb))
        p, v = reference(tb, operation)
        target_v = bv.copy() if stop_at_end and leg == len(times)-1 else bv+v[vertex]
        departure, coast, arrival, dv0, dv1, diag = solve_leg(
            driver, leg, tb-ta, br+p[vertex], target_v)
        burns.extend([dict(elapsed_s=float(ta), kind='departure', dv_mps=dv0.tolist()),
                      dict(elapsed_s=float(tb), kind='arrival', dv_mps=dv1.tolist())])
        diagnostics.append(dict(diag, elapsed_s=float(tb)))
        print(f'{label}: leg {leg}/{len(times)-1}; position error {diag["endpoint_position_error_m"]:.3g} m')
    case = dict(satellite=sat, driver=driver, vertex=LABELS[vertex],
                operation=operation, duration_s=float(duration_s), spacing_s=float(spacing_s),
                waypoints=times, burns=burns, diagnostics=diagnostics)
    (OUT/(label+'_replay.json')).write_text(json.dumps(
        {k: case[k] for k in ['vertex','operation','duration_s','spacing_s','burns','diagnostics']},
        indent=2), encoding='utf-8')
    return case


## 8. Build the selected experiments — slow step

This is the only follower-targeting step. Each experiment/spacing is a separate clip with its own spacecraft; the playback helper later hides all other clips. Using all six vertices or a full rotation increases the build time.

Wait for this cell to finish before configuring the viewer. If you change physical settings, restart the kernel and run from the top, rather than re-running this cell on an already populated scenario.

In [9]:
if RUN_SINGLE_TRANSFER:
    case = run_visual_case('single_transfer', 'Rotating baseline', SELECTED_VERTICES[0],
                           DEMO_SPACING_S, DEMO_SPACING_S, stop_at_end=False)
    clips[('single_transfer', max(BURN_SPACINGS_HOURS))] = [case]
for operation, prefix, enabled, duration in [
    ('Rotating baseline', 'rotating_baseline', RUN_ROTATING_BASELINE, TRACK_DURATION_S),
    ('Fixed baseline', 'fixed_baseline', RUN_FIXED_BASELINE, TRACK_DURATION_S),
    ('repointing', 'repointing', RUN_REPOINTING, SLEW_DURATION_S)]:
    if enabled:
        for spacing_hr in BURN_SPACINGS_HOURS:
            group = []
            for vertex in SELECTED_VERTICES:
                name = f'{prefix}_{number_tag(spacing_hr)}h_{LABELS[vertex]}'
                group.append(run_visual_case(name, operation, vertex, duration, spacing_hr*3600))
            clips[(operation, spacing_hr)] = group
all_cases = [case for group in clips.values() for case in group]
print('Completed clips:')
for key, group in clips.items():
    print(key, '| vertices:', [c['vertex'] for c in group], '| days:', group[0]['duration_s']/DAY_S)


rotating_baseline_24h_Xp: leg 1/28; position error 8.75e-05 m
rotating_baseline_24h_Xp: leg 2/28; position error 7.39e-06 m
rotating_baseline_24h_Xp: leg 3/28; position error 1.51e-05 m
rotating_baseline_24h_Xp: leg 4/28; position error 1.14e-05 m
rotating_baseline_24h_Xp: leg 5/28; position error 2.37e-05 m
rotating_baseline_24h_Xp: leg 6/28; position error 1.5e-05 m
rotating_baseline_24h_Xp: leg 7/28; position error 3.02e-06 m
rotating_baseline_24h_Xp: leg 8/28; position error 1.87e-06 m
rotating_baseline_24h_Xp: leg 9/28; position error 2.4e-06 m
rotating_baseline_24h_Xp: leg 10/28; position error 4.9e-06 m
rotating_baseline_24h_Xp: leg 11/28; position error 1.22e-06 m
rotating_baseline_24h_Xp: leg 12/28; position error 7.73e-07 m
rotating_baseline_24h_Xp: leg 13/28; position error 5.1e-06 m
rotating_baseline_24h_Xp: leg 14/28; position error 4.73e-06 m
rotating_baseline_24h_Xp: leg 15/28; position error 2.38e-07 m
rotating_baseline_24h_Xp: leg 16/28; position error 9.16e-07 m
rotat

## 9. Give spacecraft coloured paths, labels and markers

Base is white. Each follower keeps its vertex colour across experiments and spacings. The same colour is applied explicitly to its local trajectory frame as well as its marker. This avoids confusing Astrogator coast-segment colours with spacecraft identity.

At this scale markers and points are clearer than oversized spacecraft models. Model display is turned off; visible symbols do not represent physical spacecraft size. The full Earth-centred orbit line is hidden in close-ups, and ground tracks are disabled.

In [10]:
COLOURS['Yp'] = '%000255000'  # Bright green, using STK's explicit RGB format.


def cmd(command):
    try:
        return root.ExecuteCommand(command)
    except Exception as exc:
        raise RuntimeError('STK Connect failed: ' + command) from exc


def path_of(sat):
    return 'Satellite/' + sat.InstanceName


def style_satellite(sat, colour, label):
    obj = '*/' + path_of(sat)

    cmd(f'Graphics {obj} SetColor {colour}')
    cmd(f'Graphics {obj} Label Show On LabelText "{label}"')
    cmd(f'VO {obj} Model Show Off')
    cmd(f'VO {obj} Marker Show On MarkerType Line Shape Circle Size 10')
    cmd(f'VO {obj} Point Show On Size 7')
    cmd(
        f'VO {obj} Pass3D Inherit Off '
        'OrbitLead None OrbitTrail All GroundLead None GroundTrail None'
    )

    for system in ['FixedByWindow', 'InertialByWindow']:
        cmd(f'VO {obj} OrbitSystem Modify System "{system}" Show Off')

    # Reuse an existing display frame if this cell was partially run before.
    try:
        cmd(
            f'VO {obj} OrbitSystem Modify System "{LOCAL_SYSTEM}" '
            f'Color {colour} Show On'
        )
    except RuntimeError:
        cmd(
            f'VO {obj} OrbitSystem Add System "{LOCAL_SYSTEM}" '
            f'Color {colour} Show On'
        )

    cmd(f'Graphics {obj} Show Off')


style_satellite(Base, 'white', 'Base')

for case in all_cases:
    style_satellite(
        case['satellite'],
        COLOURS[case['vertex']],
        case['vertex'],
    )

cmd(f'VO * GlobeDetails Background Color black WindowID {WINDOW_ID}')
print('Colours:', COLOURS)

Colours: {'Xp': 'red', 'Xm': 'cyan', 'Yp': '%000255000', 'Ym': 'magenta', 'Zp': 'yellow', 'Zm': 'orange'}


## 10. Create optional vectors and octahedron edges

Create arrows showing the actual Base-to-spacecraft offsets, fixed guides for the initial and final requested line-of-sight directions, and optional ICRF axes. Offset arrows have their true physical length; the ICRF axes are display-scaled. The LOS guides represent the requested endpoint directions, not the instantaneous pointing during a slew.

Edge lines connect the actual spacecraft at the current animation time. Only edges between selected vertices are drawn. These lines remain available in every viewpoint, including context view; arrows remain hidden in context view. Use the display switches to choose which overlays appear.

In [11]:
vector_displays = []
edge_displays = []


def set_vector(name, definition):
    try:
        cmd(f'VectorTool * {BASE_PATH} Modify Vector {name} {definition}')
    except RuntimeError:
        cmd(f'VectorTool * {BASE_PATH} Create Vector {name} {definition}')


def register_vector(component, category, colour='white', owner=BASE_OBJECT, **extra):
    options = ' '.join(f'{key} {value}' for key, value in extra.items())
    settings = f'Show Off ShowLabel On Color {colour} Thickness 2 {options}'
    try:
        cmd(f'VO {owner} SetVectorGeometry Modify "{component}" {settings}')
    except RuntimeError:
        cmd(f'VO {owner} SetVectorGeometry Add "{component}" {settings}')
    vector_displays.append(dict(owner=owner, component=component, category=category))


cmd(f'VO {BASE_OBJECT} SetVectorGeometry Data FixVectorAxesScaleToModel Off Scale 2 VectorAxesAngleSwapDist All')

for name, direction, colour in [('Initial_LOS', LOS_START, 'cyan'), ('Final_LOS', LOS_END, 'yellow')]:
    x, y, z = 1.8 * L_M * unit(direction)
    set_vector(name, f'"Fixed in Axes" Cartesian {x:.16g} {y:.16g} {z:.16g} "CentralBody/Earth ICRF"')
    register_vector(BASE_PATH + ' ' + name + ' Vector', name, colour, UseTrueScale='On')

register_vector('CentralBody/Earth ICRF Axes', 'axes', 'white',
                OriginPointDef='"' + BASE_PATH + ' Center"')

for key, group in clips.items():
    for case in group:
        name = 'Offset_' + case['satellite'].InstanceName
        destination = path_of(case['satellite'])
        set_vector(name, f'"Displacement" "{BASE_PATH} Center" "{destination} Center"')
        register_vector(BASE_PATH + ' ' + name + ' Vector', key, COLOURS[case['vertex']],
                        UseTrueScale='On')

    for a, b in combinations(group, 2):
        ia, ib = LABELS.index(a['vertex']), LABELS.index(b['vertex'])
        if np.isclose(np.linalg.norm(VERTICES[ia] - VERTICES[ib]), np.sqrt(2) * L_M):
            pa, pb = path_of(a['satellite']), path_of(b['satellite'])
            settings = f'FromObj {pa} ToObj {pb} Show Off Color gray60 LineWidth 1 WindowID {WINDOW_ID}'
            try:
                cmd(f'VO * ObjectLine Modify {settings}')
            except RuntimeError:
                cmd(f'VO * ObjectLine Add {settings}')
            edge_displays.append(dict(key=key, a=pa, b=pb))

ACTIVE_KEY = globals().get('ACTIVE_KEY', None)
CURRENT_VIEW = globals().get('CURRENT_VIEW', 'oblique')


def apply_overlays():
    for item in vector_displays:
        category = item['category']
        show = SHOW_VECTORS and (
            (category == 'Initial_LOS' and SHOW_LOS_GUIDES)
            or (category == 'Final_LOS' and SHOW_LOS_GUIDES and ACTIVE_KEY is not None and ACTIVE_KEY[0] == 'repointing')
            or (category == 'axes' and SHOW_ICRF_AXES)
            or (isinstance(category, tuple) and category == ACTIVE_KEY and SHOW_OFFSET_VECTORS))
        if CURRENT_VIEW == 'context':
            show = False
        cmd(f'VO {item["owner"]} SetVectorGeometry Modify "{item["component"]}" Show {"On" if show else "Off"}')

    for item in edge_displays:
        show = SHOW_EDGES and item['key'] == ACTIVE_KEY
        cmd(f'VO * ObjectLine Modify FromObj {item["a"]} ToObj {item["b"]} Show {"On" if show else "Off"} WindowID {WINDOW_ID}')


apply_overlays()

### 10A. Make the arrows thinner

Set the existing vectors and axes to a thickness of one pixel to reduce visual clutter. Change `ARROW_THICKNESS` to adjust their thickness; the minimum supported value is 1. This changes only their appearance and does not rerun the propagation.

In [12]:
ARROW_THICKNESS = 1

for item in vector_displays:
    cmd(f'VO {item["owner"]} SetVectorGeometry Modify "{item["component"]}" Thickness {ARROW_THICKNESS}')

cmd('Animate * Refresh')

### 10B. Choose the displayed spacecraft names

Edit `DISPLAY_NAMES` to choose the names shown beside the spacecraft, and `BASE_DISPLAY_NAME` to change Base's displayed label. These labels apply across all experiment clips. The internal vertex identifiers and Object Browser names are preserved, and no propagation needs rerunning.

In [13]:
DISPLAY_NAMES = {
    'Xp': 'SC1',
    'Xm': 'SC2',
    'Yp': 'SC3',
    'Ym': 'SC4',
    'Zp': 'collector',
    'Zm': 'SC5',
}
BASE_DISPLAY_NAME = 'Base'

cmd(f'Graphics {BASE_OBJECT} Label LabelText "{BASE_DISPLAY_NAME}"')

for case in all_cases:
    obj = '*/' + path_of(case['satellite'])
    name = DISPLAY_NAMES[case['vertex']]
    cmd(f'Graphics {obj} Label LabelText "{name}"')

cmd('Animate * Refresh')

### 10C. Show or hide Base

Set `SHOW_BASE = True` to display Base, or `False` to hide its graphics. Base remains the reference spacecraft for the dynamics and camera, even when hidden. No propagation needs rerunning.

In [31]:
SHOW_BASE = False

cmd(f'Graphics {BASE_OBJECT} Show {"On" if SHOW_BASE else "Off"}')
cmd('Animate * Refresh')

### 10D. Show or hide offset vectors and shorten their names

Set `SHOW_OFFSET_VECTORS` to show or hide the Base-to-spacecraft arrows, and `SHOW_OFFSET_LABELS` to control their labels separately.

Set `SHORT_OFFSET_NAMES = True` and edit `OFFSET_NAMES` to use shorter vector names. A suffix such as `_C1` identifies the experiment clip and keeps the names unique. Set it to `False` to restore the original names.

Rerun this cell after changing these settings. The existing vector switches still apply, and arrows remain hidden in context view. No propagation needs rerunning.

In [30]:
SHOW_OFFSET_VECTORS = True
SHOW_OFFSET_LABELS = False
SHORT_OFFSET_NAMES = False

OFFSET_NAMES = DISPLAY_NAMES # Can also name them something different than the display names

for item in vector_displays:
    item.setdefault('original_component', item['component'])

for clip_number, (key, group) in enumerate(clips.items(), start=1):
    for case in group:
        original_name = 'Offset_' + case['satellite'].InstanceName
        original_component = BASE_PATH + ' ' + original_name + ' Vector'
        item = next(v for v in vector_displays
                    if v['original_component'] == original_component)

        name = OFFSET_NAMES[case['vertex']] + f'_C{clip_number}'
        if not SHORT_OFFSET_NAMES:
            name = original_name

        component = BASE_PATH + ' ' + name + ' Vector'

        if item['component'] != component:
            cmd(f'VectorTool * Rename "{item["component"]}" {name}')
            item['component'] = component

        cmd(f'VO {item["owner"]} SetVectorGeometry Modify "{component}" ShowLabel {"On" if SHOW_OFFSET_LABELS else "Off"}')

apply_overlays()
cmd('Animate * Refresh')

## 11. Define cameras and playback controls

`show_clip()` isolates a clip, sets its start/end times, resets to its release, chooses a camera, and starts playback. `set_view()` changes only the camera and path frame, preserving playback time. `replay_leg()` selects one coast between consecutive waypoints, so you can watch an individual transfer rather than the whole experiment. Coincident arrival/departure impulses remain at the same instant; no artificial delay is inserted.

Camera presets are **oblique**, **along_los** (from the positive initial-LOS side looking at Base), **side** (from the initial formation x direction), and **context** (Earth-centred). The LOS camera stays aligned with the initial LOS during a slew, which makes the tilt visible. The context view shows where the cluster travels; kilometre-scale separations cannot be resolved at that scale.

In [16]:
def utc(epoch):
    return root.ConversionUtility.ConvertDate('EpSec', 'UTCG', str(float(epoch)))


def clip_key(operation, spacing_hours=None):
    choices = [key for key in clips if key[0] == operation]
    if not choices:
        raise ValueError(f'{operation} was not built. Available clips: {list(clips)}')
    if spacing_hours is None:
        return choices[0]
    for key in choices:
        if np.isclose(key[1], float(spacing_hours)):
            return key
    raise ValueError(f'No {spacing_hours} h clip for {operation}. Choices: {choices}')


def set_playback(start_s, stop_s, play=True):
    if ACTIVE_KEY is None:
        raise ValueError('Select a clip first.')

    duration = clips[ACTIVE_KEY][0]['duration_s']
    if not (0 <= start_s < stop_s <= duration + 1e-6):
        raise ValueError('Choose an interval within the active clip.')

    cmd('Animate * Pause')
    cmd(f'SetAnimation * StartAndCurrentTime "{utc(T0 + start_s)}"')
    cmd(f'SetAnimation * EndTime "{utc(T0 + stop_s)}"')
    cmd('SetAnimation * AnimationMode Normal')

    step = (stop_s - start_s) / (PLAYBACK_SECONDS * PLAYBACK_FPS)
    cmd(f'SetAnimation * TimeStep {step:.12g} RefreshDelta {1 / PLAYBACK_FPS:.12g} RefreshMode RefreshDelta')
    cmd(f'SetAnimation * EndMode {"Loop" if LOOP_PLAYBACK else "End"}')
    cmd('Animate * Reset')

    print('Playback:', ACTIVE_KEY, '|', utc(T0 + start_s), 'to', utc(T0 + stop_s), 'UTC')
    print('Simulation step:', round(step, 3), 's/frame')
    if play:
        cmd(f'Animate * Start Forward {"Loop" if LOOP_PLAYBACK else "End"}')


def show_clip(operation, spacing_hours=None, view='oblique', play=True):
    global ACTIVE_KEY

    key = clip_key(operation, spacing_hours)
    cmd('Animate * Pause')
    ACTIVE_KEY = key
    visible = {c['satellite'].InstanceName for c in clips[key]}

    for sat in [Base] + [c['satellite'] for c in all_cases]:
        on = sat is Base or sat.InstanceName in visible
        obj = '*/' + path_of(sat)
        cmd(f'Graphics {obj} Show {"On" if on else "Off"}')
        cmd(f'Graphics {obj} Label Show {"On" if SHOW_LABELS else "Off"}')

    # Put Base inside its ephemeris interval before centring the camera.
    cmd(f'SetAnimation * CurrentTime "{utc(T0)}"')
    set_view(view)
    set_playback(0.0, clips[key][0]['duration_s'], play)
    print('Visible:', {c['vertex']: COLOURS[c['vertex']] for c in clips[key]}, '| Base: white')


def replay_leg(leg=1, play=True):
    if ACTIVE_KEY is None:
        raise ValueError('Select a clip first.')

    times = clips[ACTIVE_KEY][0]['waypoints']
    if type(leg) is not int or not 1 <= leg < len(times):
        raise ValueError(f'Choose a leg from 1 to {len(times) - 1}.')

    set_playback(float(times[leg - 1]), float(times[leg]), play)
    print('Leg', leg, '| endpoint impulses are instantaneous.')


def jump_to(elapsed_hours):
    if ACTIVE_KEY is None:
        raise ValueError('Select a clip first.')

    t = float(elapsed_hours) * 3600
    if not np.isfinite(t) or not 0 <= t <= clips[ACTIVE_KEY][0]['duration_s']:
        raise ValueError('Requested time is outside the clip.')

    cmd('Animate * Pause')
    cmd(f'SetAnimation * CurrentTime "{utc(T0 + t)}"')
    cmd('Animate * Refresh')

### 11A. Formation context view

Context view follows Base at the same angle and distance as the oblique view. Each spacecraft leaves a coloured trail showing its recent Earth-inertial orbital motion.

Change `CONTEXT_TRAIL_SECONDS` to adjust the trail length. Start with 5 seconds: even a few seconds of orbital motion can span kilometres at this zoom.

Thin arrows show the directions from Base toward Earth and toward the Sun–Earth–Moon L2 point. Their displayed lengths do not represent distances. The arrows remain available when Base's marker is hidden.

In [32]:
from agi.stk13.stkobjects import AgESTKObjectType

CONTEXT_ORBIT_TRAIL_HOURS = 25.0
CONTEXT_ROTATION_TRAIL_HOURS = 24*10   # 28 days for one rotation.
# Either duration: None = entire history; 0 = hide that trail.

SHOW_CONTEXT_VECTORS = True
SHOW_EARTH_DIRECTION = True
SHOW_L2_DIRECTION = True
CONTEXT_ARROW_THICKNESS = 1


def update_component(parent, kind, name, definition):
    try:
        cmd(f'VectorTool * {parent} Modify {kind} {name} {definition}')
    except RuntimeError:
        cmd(f'VectorTool * {parent} Create {kind} {name} {definition}')


# Define the moving L2 point and the two direction vectors.
update_component('CentralBody/Sun', 'Point', 'Viewer_L2',
                 '"Libration" Sun L2 Earth Moon')

update_component(BASE_PATH, 'Vector', 'To_Earth',
                 f'"Displacement" "{BASE_PATH} Center" "CentralBody/Earth Center"')

update_component(BASE_PATH, 'Vector', 'To_L2',
                 f'"Displacement" "{BASE_PATH} Center" "CentralBody/Sun Viewer_L2"')

# Display on the scenario so hiding Base does not hide these arrows.
context_categories = ['Context_Earth', 'Context_L2']
vector_displays[:] = [item for item in vector_displays
                      if item['category'] not in context_categories]

for name, category, colour in [('To_Earth', 'Context_Earth', 'cyan'),
                               ('To_L2', 'Context_L2', 'yellow')]:
    component = BASE_PATH + ' ' + name + ' Vector'
    settings = (
        f'Show Off ShowLabel On Color {colour} '
        f'Thickness {CONTEXT_ARROW_THICKNESS} UseTrueScale Off '
        f'OriginPointDef "{BASE_PATH} Center" WindowID {WINDOW_ID}')
    try:
        cmd(f'VO * SetVectorGeometry Modify "{component}" {settings}')
    except RuntimeError:
        cmd(f'VO * SetVectorGeometry Add "{component}" {settings}')
    vector_displays.append(dict(owner='*', component=component, category=category))

cmd(f'VO * SetVectorGeometry Data Scale 1 VectorAxesAngleSwapDist All WindowID {WINDOW_ID}')


def apply_overlays():
    context = CURRENT_VIEW == 'context'

    for item in vector_displays:
        category = item['category']

        if category == 'Context_Earth':
            show = SHOW_VECTORS and context and SHOW_CONTEXT_VECTORS and SHOW_EARTH_DIRECTION
        elif category == 'Context_L2':
            show = SHOW_VECTORS and context and SHOW_CONTEXT_VECTORS and SHOW_L2_DIRECTION
        else:
            show = SHOW_VECTORS and not context and (
                (category == 'Initial_LOS' and SHOW_LOS_GUIDES)
                or (category == 'Final_LOS' and SHOW_LOS_GUIDES and ACTIVE_KEY is not None and ACTIVE_KEY[0] == 'repointing')
                or (category == 'axes' and SHOW_ICRF_AXES)
                or (isinstance(category, tuple) and category == ACTIVE_KEY and SHOW_OFFSET_VECTORS))

        cmd(f'VO {item["owner"]} SetVectorGeometry Modify "{item["component"]}" Show {"On" if show else "Off"}')

    for item in edge_displays:
        show = SHOW_EDGES and tuple(item['key']) == ACTIVE_KEY
        cmd(f'VO * ObjectLine Modify FromObj {item["a"]} ToObj {item["b"]} Show {"On" if show else "Off"} WindowID {WINDOW_ID}')


# Create display-only copies for independently timed Base-relative trails.
# These load the existing ephemerides; they do not rerun the targeter.
trail_folder = (OUT / 'visual_trails').resolve()
trail_folder.mkdir(parents=True, exist_ok=True)

for case in all_cases:
    source = case['satellite']
    name = 'VizTrail_' + source.InstanceName
    obj = '*/Satellite/' + name

    try:
        helper = root.GetObjectFromPath('Satellite/' + name)
    except RuntimeError:
        ephemeris = trail_folder / (name + '.e')
        cmd(f'ExportDataFile */{path_of(source)} Ephemeris "{ephemeris}" Type STK CoordSys ICRF CentralBody Earth InterpBoundaries Include TimeSteps UseEphemerisSteps')
        helper = scenario.Children.New(AgESTKObjectType.eSatellite, name)
        cmd(f'SetState {obj} FromFile "{ephemeris}" Save Locally')

    case['rotation_trail'] = helper
    colour = COLOURS[case['vertex']]

    cmd(f'Graphics {obj} Show Off')
    cmd(f'Graphics {obj} SetColor {colour}')
    cmd(f'Graphics {obj} Label Show Off')
    cmd(f'VO {obj} Model Show Off')
    cmd(f'VO {obj} Marker Show Off')
    cmd(f'VO {obj} Point Show Off')

    for system in ['FixedByWindow', 'InertialByWindow']:
        cmd(f'VO {obj} OrbitSystem Modify System "{system}" Show Off')

    try:
        cmd(f'VO {obj} OrbitSystem Modify System "{LOCAL_SYSTEM}" Color {colour} Show On')
    except RuntimeError:
        cmd(f'VO {obj} OrbitSystem Add System "{LOCAL_SYSTEM}" Color {colour} Show On')


def trail_option(hours):
    if hours is None:
        return 'All'
    hours = float(hours)
    if not np.isfinite(hours) or hours < 0:
        raise ValueError('Trail hours must be non-negative, or None.')
    return 'None' if hours == 0 else f'Time {hours * 3600:.12g}'


def set_paths(mode=None):
    if mode is None:
        mode = TRAIL_MODE
    if mode not in ['history', 'full']:
        raise ValueError("Use 'history' or 'full'.")

    orbit_trail = trail_option(CONTEXT_ORBIT_TRAIL_HOURS)
    rotation_trail = trail_option(CONTEXT_ROTATION_TRAIL_HOURS)
    context = CURRENT_VIEW == 'context'

    for case in all_cases:
        source = case['satellite']
        obj = '*/' + path_of(source)

        if context:
            cmd(f'VO {obj} Pass3D Inherit Off OrbitLead None OrbitTrail {orbit_trail}')
        else:
            cmd(f'VO {obj} Pass3D Inherit Off OrbitLead {"All" if mode == "full" else "None"} OrbitTrail All')

        helper_obj = '*/' + path_of(case['rotation_trail'])
        show = context and source.Graphics.IsObjectGraphicsVisible and rotation_trail != 'None'
        cmd(f'Graphics {helper_obj} Show {"On" if show else "Off"}')
        cmd(f'VO {helper_obj} Pass3D Inherit Off OrbitLead None OrbitTrail {rotation_trail}')


def set_view(view='oblique', range_m=None):
    global CURRENT_VIEW

    if view not in ['oblique', 'along_los', 'side', 'context']:
        raise ValueError('Unknown camera: ' + view)

    distance = CAMERA_RANGE_M if range_m is None else float(range_m)
    if not np.isfinite(distance) or distance <= 0:
        raise ValueError('Camera range must be positive and finite.')

    CURRENT_VIEW = view
    context = view == 'context'
    colours = {c['satellite'].InstanceName: COLOURS[c['vertex']] for c in all_cases}

    for sat in [Base] + [c['satellite'] for c in all_cases]:
        obj = '*/' + path_of(sat)
        local_show = 'On' if sat is not Base and not context else 'Off'
        inertial_show = 'On' if sat is not Base and context else 'Off'
        colour = 'white' if sat is Base else colours[sat.InstanceName]
        cmd(f'VO {obj} OrbitSystem Modify System "{LOCAL_SYSTEM}" Show {local_show}')
        cmd(f'VO {obj} OrbitSystem Modify System "InertialByWindow" Color {colour} Show {inertial_show}')

    set_paths()
    cmd(f'VO * ViewFromTo Normal From {BASE_PATH} To {BASE_PATH} WindowID {WINDOW_ID}')
    cmd(f'VO * ViewFromTo Reset WindowID {WINDOW_ID}')

    axes = 'CentralBody/Earth ICRF Axes'
    select_frame = f'VO * View SetReferenceFrame "{axes}" WindowID {WINDOW_ID}'
    try:
        cmd(select_frame)
    except RuntimeError:
        cmd(f'VO * View AddReferenceFrame "{axes}" Object {BASE_PATH}')
        cmd(select_frame)

    cmd(f'VO * ViewFromTo Parameters UseUpVector Off LockDirection Off WindowID {WINDOW_ID}')

    direction = {
        'oblique': unit(Q0 @ np.array([1.5, -2.0, 1.3])),
        'context': unit(Q0 @ np.array([1.5, -2.0, 1.3])),
        'along_los': unit(LOS_START),
        'side': Q0[:, 0],
    }[view]

    dec = np.rad2deg(np.arcsin(np.clip(direction[2], -1, 1)))
    ra = np.rad2deg(np.arctan2(direction[1], direction[0])) % 360
    cmd(f'VO * ViewerPosition {dec:.12g} {ra:.12g} {distance:.12g} {WINDOW_ID}')
    apply_overlays()
    cmd('Animate * Refresh')


set_view('context')

## 12. Start the first available experiment

This chooses the Rotating baseline when enabled, otherwise the first available clip. It shows only the selected followers, follows Base and starts at release. Bring STK's **3D Graphics 1** window to the front. No waiting or Python animation loop is needed; STK plays independently after this cell returns.

With a one-day arc, the 28-day rotation covers only about 12.9°. For a complete revolution, change `TRACK_DURATION_DAYS` to `None` before rebuilding. With only three followers the octahedron is necessarily incomplete.

In [18]:
def set_view(view='oblique', range_m=None):
    global CURRENT_VIEW

    if view not in ['oblique', 'along_los', 'side', 'context']:
        raise ValueError('Unknown camera: ' + view)

    distance = CAMERA_RANGE_M if range_m is None else float(range_m)
    if not np.isfinite(distance) or distance <= 0:
        raise ValueError('Camera range must be positive and finite.')

    CURRENT_VIEW = view
    context = view == 'context'

    for sat in [Base] + [c['satellite'] for c in all_cases]:
        obj = '*/' + path_of(sat)
        local_show = 'Off' if context or sat is Base else 'On'
        inertial_show = 'On' if context else 'Off'
        cmd(f'VO {obj} OrbitSystem Modify System "{LOCAL_SYSTEM}" Show {local_show}')
        cmd(f'VO {obj} OrbitSystem Modify System "InertialByWindow" Show {inertial_show}')

    anchor = 'CentralBody/Earth' if context else BASE_PATH
    cmd(f'VO * ViewFromTo Normal From {anchor} To {anchor} WindowID {WINDOW_ID}')

    # Register ICRF for this view class if it is not already selectable.
    axes = 'CentralBody/Earth ICRF Axes'
    select_frame = f'VO * View SetReferenceFrame "{axes}" WindowID {WINDOW_ID}'
    try:
        cmd(select_frame)
    except RuntimeError:
        target = 'CentralBody' if context else BASE_PATH
        cmd(f'VO * View AddReferenceFrame "{axes}" Object {target}')
        cmd(select_frame)

    cmd(f'VO * ViewFromTo Parameters UseUpVector Off LockDirection Off WindowID {WINDOW_ID}')

    direction = {
        'oblique': unit(Q0 @ np.array([1.5, -2.0, 1.3])),
        'along_los': unit(LOS_START),
        'side': Q0[:, 0],
        'context': unit(np.array([1.0, -1.0, 0.7])),
    }[view]

    if context and range_m is None:
        distance = 3.0 * np.linalg.norm(base_state(float(T0))[0])

    dec = np.rad2deg(np.arcsin(np.clip(direction[2], -1, 1)))
    ra = np.rad2deg(np.arctan2(direction[1], direction[0])) % 360

    cmd(f'VO * ViewFromTo Reset WindowID {WINDOW_ID}')
    cmd(f'VO * ViewerPosition {dec:.12g} {ra:.12g} {distance:.12g} {WINDOW_ID}')
    apply_overlays()

In [19]:
FIRST_KEY = next((key for key in clips if key[0] == 'Rotating baseline'), next(iter(clips)))
show_clip(*FIRST_KEY, view='oblique', play=AUTO_PLAY)


Playback: ('Rotating baseline', 24.0) | 6 Oct 2022 07:10:00.000 to 3 Nov 2022 07:10:00.000 UTC
Simulation step: 3024.0 s/frame
Visible: {'Xp': 'red', 'Xm': 'cyan', 'Yp': '%000255000', 'Ym': 'magenta', 'Zp': 'yellow', 'Zm': 'orange'} | Base: white


## 13. Look along the initial line of sight

Run this cell whenever you want a face-on view. It changes the viewpoint without resetting time. In the rotating experiment the equatorial spacecraft move around the axis; `Zp` and `Zm`, if included, project close to Base and may overlap in this view. Use the next side view to separate them.

In [20]:
set_view('along_los')


## 14. Side and oblique views

The side view exposes axial separation and the out-of-plane tilt during re-pointing. The oblique view is often easiest for seeing the full three-dimensional geometry. Run either line by itself to choose a view; as written, this cell ends in the side view. A larger `range_m` zooms out.

In [21]:
set_view('oblique')
set_view('side', range_m=CAMERA_RANGE_M)


## 15. Replay each experiment from its own start

Run the line for the experiment you want to watch. `None` selects its first built spacing; replace it with `3.0` or `6.0` to choose explicitly. This cell runs the first available item in `WATCH_OPERATION`; change that setting and re-run the cell to switch. The single-transfer case contains only the first selected follower. Every switch resets the animation interval and hides the previous clip.

For the Fixed baseline, offset arrows and edges are more informative than path length: its target offsets stay constant in this inertial display. Small deviations between burns are real propagated motion.

In [22]:
WATCH_OPERATION = 'Rotating baseline'           # 'Rotating baseline', 'Fixed baseline', 'single_transfer', 'repointing'
WATCH_SPACING_HOURS = None
if any(key[0] == WATCH_OPERATION for key in clips):
    show_clip(WATCH_OPERATION, WATCH_SPACING_HOURS, view='oblique', play=True)
else:
    print('That experiment was disabled. Available:', list(clips))


Playback: ('Rotating baseline', 24.0) | 6 Oct 2022 07:10:00.000 to 3 Nov 2022 07:10:00.000 UTC
Simulation step: 3024.0 s/frame
Visible: {'Xp': 'red', 'Xm': 'cyan', 'Yp': '%000255000', 'Ym': 'magenta', 'Zp': 'yellow', 'Zm': 'orange'} | Base: white


In [23]:
# Other examples, each independently runnable after uncommenting:
# show_clip('Rotating baseline', 3.0, view='along_los')
# show_clip('Fixed baseline', 6.0, view='oblique')
#show_clip('single_transfer', view='oblique')

## 16. Watch a single burn-to-burn leg or inspect a chosen instant

Change `LEG_TO_WATCH` to an available leg. Playback begins exactly at that leg's departure and stops/loops at its arrival. Existing trail history can include earlier legs; the animated time interval is only the chosen coast. Use `set_paths('full')` if you want its future trajectory visible too.

To inspect an instant, run `jump_to(hours)` separately; it pauses playback. Arrival and departure at a shared waypoint happen at the same time, so their velocity changes are not shown as two visually separated events.

In [24]:
LEG_TO_WATCH = 1
replay_leg(LEG_TO_WATCH)
# jump_to(3.0)                          # Hours since release, within the selected clip.
# set_paths('full')
# set_paths('history')


Playback: ('Rotating baseline', 24.0) | 6 Oct 2022 07:10:00.000 to 7 Oct 2022 07:10:00.000 UTC
Simulation step: 108.0 s/frame
Leg 1 | endpoint impulses are instantaneous.


## 17. Place the formation in orbital context

This switches to an Earth-centred ICRF camera and shows the actual inertial orbit paths. The entire cluster is unresolved at this scale; it can appear as one spacecraft. Return to a Base-centred close-up with the commented line. Optional close-up arrows and edges are hidden in context view and restored when you return.

In [25]:
#set_view('context')                    # Default: 20 km from Base.
set_view('context', range_m=5000)      # Closer
#set_view('oblique', range_m=5000)     # Formation close-up

## 18. Toggle vectors, edges and full paths

Set `SHOW_VECTORS = False` for a clean view, or enable only the individual arrows you need. Offset arrows show true spacecraft separation. LOS guides show requested endpoint directions, not a measured instantaneous optical axis. ICRF axes are a display aid with no physical length scale. These settings do not change dynamics or re-run Astrogator.

In [26]:
# Edit these switches, then re-run this cell.
# SHOW_VECTORS = False
# SHOW_OFFSET_VECTORS = True
# SHOW_LOS_GUIDES = True
# SHOW_ICRF_AXES = False
# SHOW_EDGES = False
# SHOW_LABELS = True
# TRAIL_MODE = 'full'
apply_overlays()
set_paths()
for sat in [Base]+[c['satellite'] for c in all_cases]:
    cmd(f'Graphics */{path_of(sat)} Label Show {"On" if SHOW_LABELS else "Off"}')


## 19. Pause, resume or reset playback

Select one action and re-run this cell. Reset uses the currently selected animation interval, including a single-leg replay. To return to the whole experiment, call `show_clip(*ACTIVE_KEY)`. To change playback speed, edit `PLAYBACK_SECONDS` in section 1 and replay the clip or leg; it does not require re-propagating.

In [27]:
ACTION = 'pause'                         # 'pause', 'resume', 'reset', 'whole_clip'
if ACTION == 'pause': cmd('Animate * Pause')
elif ACTION == 'resume': cmd(f'Animate * Start Forward {"Loop" if LOOP_PLAYBACK else "End"}')
elif ACTION == 'reset': cmd('Animate * Reset')
elif ACTION == 'whole_clip': show_clip(*ACTIVE_KEY, view=CURRENT_VIEW)
else: raise ValueError('Unknown action.')


## 20. Save the scenario and leave a useful clip playing

Save a checkpoint with all the propagated experiments, graphics settings and the default close-up view. `SaveAs` writes STK's scenario and companion files into the new results folder; keep that folder together when moving the scenario. The notebook itself remains self-contained and can recreate it.

This final cell also makes Run All end with the first available experiment playing at release, rather than leaving the viewer paused in the context view. To preserve a different view, pause there and save from STK or call `root.SaveScenario()` after this initial save.

In [28]:
show_clip(*FIRST_KEY, view='oblique', play=False)
if SAVE_SCENARIO:
    checkpoint = OUT/'LIFE_STK_visualisation.sc'
    root.SaveAs(str(checkpoint))
    print('Saved:', checkpoint)
if AUTO_PLAY:
    cmd(f'Animate * Start Forward {"Loop" if LOOP_PLAYBACK else "End"}')


Playback: ('Rotating baseline', 24.0) | 6 Oct 2022 07:10:00.000 to 3 Nov 2022 07:10:00.000 UTC
Simulation step: 3024.0 s/frame
Visible: {'Xp': 'red', 'Xm': 'cyan', 'Yp': '%000255000', 'Ym': 'magenta', 'Zp': 'yellow', 'Zm': 'orange'} | Base: white
Saved: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Own-Code\First-Experiment\results\LIFE_STK_visualisation_20261102_233558_234649\LIFE_STK_visualisation.sc


### 20A. Save the notebook controls for later reuse

Save the current scenario and a companion `viewer_state.json` file containing the experiment groups, waypoint times, camera settings and overlay references.

Keep this file in the scenario folder. Together with the saved scenario and ephemerides, it allows the notebook controls to be restored without rerunning the experiments.

In [29]:
settings = [
    'WINDOW_ID', 'CAMERA_RANGE_M', 'PLAYBACK_SECONDS', 'PLAYBACK_FPS',
    'LOOP_PLAYBACK', 'AUTO_PLAY', 'TRAIL_MODE', 'SHOW_VECTORS',
    'SHOW_OFFSET_VECTORS', 'SHOW_LOS_GUIDES', 'SHOW_ICRF_AXES',
    'SHOW_EDGES', 'SHOW_LABELS', 'COLOURS',
]

viewer_state = {
    'scenario_name': scenario.InstanceName,
    'base_path': BASE_PATH,
    'local_system': LOCAL_SYSTEM,
    'T0': float(T0),
    'Q0': np.asarray(Q0).tolist(),
    'LOS_START': np.asarray(LOS_START).tolist(),
    'settings': {name: globals()[name] for name in settings},
    'SHOW_BASE': globals().get('SHOW_BASE', True),
    'CURRENT_VIEW': CURRENT_VIEW,
    'vector_displays': vector_displays,
    'edge_displays': edge_displays,
    'clips': [],
}

for key, group in clips.items():
    cases = []
    for case in group:
        cases.append({
            'satellite_path': path_of(case['satellite']),
            'vertex': case['vertex'],
            'operation': case['operation'],
            'duration_s': float(case['duration_s']),
            'spacing_s': float(case['spacing_s']),
            'waypoints': np.asarray(case['waypoints']).tolist(),
        })
    viewer_state['clips'].append({'key': list(key), 'cases': cases})

root.SaveScenario()
state_file = OUT / 'viewer_state.json'
state_file.write_text(json.dumps(viewer_state, indent=2), encoding='utf-8')
print('Saved viewer setup:', state_file)

Saved viewer setup: c:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Own-Code\First-Experiment\results\LIFE_STK_visualisation_20261102_233558_234649\viewer_state.json


## If the view is unclear

- **Only one dot:** run `set_view('oblique')`. An Earth-centred camera cannot resolve a kilometre-scale cluster. The face-on camera can also project axial vertices onto Base.
- **Not much rotation:** the default arc is one day, about 12.9° of the configured rotation. Increase `TRACK_DURATION_DAYS` before rebuilding for a longer arc.
- **No trail at the start:** history mode draws the trail as time advances. Use `set_paths('full')` to see the complete propagated arc immediately. A fixed baseline can have an extremely short path in the local frame.
- **No movement in STK:** bring the 3D window to the front; use `show_clip(*ACTIVE_KEY)` to reset the correct interval. Check that the animation toolbar is advancing. `WINDOW_ID` must match an open 3D window.
- **Small path bends or deviations:** the spacecraft coast freely between targeted endpoints; the analytical reference is not imposed continuously. No analytical reference satellites are mixed into the propagated formation.
- **Targeting fails:** keep the printed failed-leg information. The notebook stops at the source solver's position/velocity checks. It does not silently substitute ideal curves.
- **Build feels slow:** section 8 runs many Astrogator corrections, including re-running each spacecraft's accumulating sequence. For a quick preview, use one spacing and fewer selected followers before restarting.

This notebook's code and reference geometry have been checked without executing STK. Windows STK playback and your installed force-model component must be verified on your machine.

## Sources

The experiment's physical inputs, geometry, spacecraft constructors and differential-correction solver come from the attached `LIFE_formation_first_steps(4).ipynb`. The seed state is reused at its original precision; no JWST ephemeris is inferred or downloaded.

Official Ansys STK documentation consulted for the visual controls:

- [VectorTool System](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VectorToolSystem.htm) — assembled Base-centred inertial system.
- [VectorTool Vector](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VectorToolVector.htm) — displacement and fixed-in-axes arrows.
- [VO OrbitSystem](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VOOrbitSystem.htm) — relative and inertial path drawing.
- [VO Pass3D](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VOPass3D.htm) — full paths and trail history.
- [VO ViewFromTo](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VOViewFromTo.htm), [VO View](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VOView.htm), [VO ViewerPosition](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VOViewerPosition.htm) — cameras and explicit range.
- [VO SetVectorGeometry](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VOSetVectorGeometry.htm) and [VO ObjectLine](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VOObjectLine.htm) — optional arrows and edges.
- [Astrogator MCS options](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSOptions.html) — disable temporary targeter graphics and segment colours.
- [SetAnimation](https://help.agi.com/stkdevkit/12.4.0/Subsystems/connectCmds/Content/cmd_SetAnimation.htm) and [Animate](https://help.agi.com/stkdevkit/Subsystems/connectCmds/Content/cmd_Animate.htm) — clip intervals, playback and pause.
- [Units_SetConnect](https://help.agi.com/stkdevkit/Subsystems/connectCmds/Content/cmd_Units_SetConnect.htm), [Graphics SetColor](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_GraphicsSetColor.htm), [Graphics Show](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_GraphicsShow.htm), [Graphics Label](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_GraphicsLabel.htm), [VO Marker](https://help.agi.com/stk/SubSystems/connectCmds/Content/cmd_VOMarker.htm), [VO Model](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VOModel.htm).